In [1]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [2]:
import bs4
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
chat_model = ChatNVIDIA(
  model="meta/llama-3.1-405b-instruct",
  api_key=os.environ["NVIDIA_API_KEY"], 
  temperature=0.0,
)

In [4]:
loader = WebBaseLoader(web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
                       bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))))

docs = loader.load()

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splits = text_splitter.split_documents(docs)

In [6]:
from langchain_chroma import Chroma
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_mongodb import MongoDBAtlasVectorSearch
from pymongo import MongoClient
from uuid import uuid4

# embedding = NVIDIAEmbeddings()
# dimension = len(embedding.embed_query("Testing embedding dimension"))
# mongo_client = MongoClient(os.environ["MONGODB_ATLAS_CLUSTER_URI"])

# DB_NAME = "langchain_test_db"
# COLLECTION_NAME = "langchain_test_vectorstores"
# ATLAS_VECTOR_SEARCH_INDEX_NAME = "langchain-test-index-vectorstores"

# MONGODB_COLLECTION = mongo_client[DB_NAME][COLLECTION_NAME]

vector_db = Chroma.from_documents(documents=splits, embedding=NVIDIAEmbeddings())
# vector_store = MongoDBAtlasVectorSearch(collection=MONGODB_COLLECTION, embedding=embedding,
#                                         index_name=ATLAS_VECTOR_SEARCH_INDEX_NAME, relevance_score_fn="cosine")

# vector_store.create_vector_search_index(dimensions=dimension)

# uuids = [str(uuid4()) for _ in range(len(splits))]
# vector_store.add_documents(documents=splits, ids=uuids)

In [7]:
retriever = vector_db.as_retriever(search_type="mmr",
                                   search_kwargs={'k': 3, 'fetch_k': 5})


prompt_template_str = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""

prompt_template = ChatPromptTemplate.from_template(prompt_template_str)

In [8]:
prompt_template.input_variables

['context', 'question']

In [9]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [10]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | chat_model
    | StrOutputParser()
)

In [11]:
rag_chain.invoke("What is Task Decomposition?")

'Task Decomposition is a process of breaking down a complex task into smaller, simpler steps. This is achieved through techniques such as Chain of Thought (CoT), which instructs a model to "think step by step" to decompose hard tasks into manageable ones. This process enables the model to utilize more test-time computation and provides insight into its thinking process.'

In [12]:
# vector_db.delete_collection()